# Export CLAP (general-audio) for on-device iOS text→music retrieval

Exports `laion/clap-htsat-unfused` — the winner of the `music_retrieval_comparison` benchmark on
size/speed — to CoreML `.mlpackage` for the retrieval flow: embed a music library once with the
**audio encoder**, embed a text query with the **text encoder**, rank tracks by cosine similarity.

Both encoders are validated against the original PyTorch model (cosine similarity of exported vs.
reference outputs) before being written to `exported_models/`.

## Why CLAP needs a very different export strategy than CLIP

CLIP's encoders are plain feed-forward transformers over pixels/tokens — trace + `coremltools.convert`
just works. CLAP's **audio** encoder is a much rougher ride:

1. **Mel-spectrogram front end.** `ClapFeatureExtractor` computes a log-mel spectrogram in NumPy
   (STFT via `np.fft.rfft`, a Slaney mel filterbank, dB compression) *before* the model ever sees the
   audio — that preprocessing normally has to be reimplemented in Swift. Instead, this notebook
   reimplements the exact same math as a `Conv1d`-based STFT (the DFT is a fixed linear operator, so a
   real/imaginary cosine–sine basis becomes a `Conv1d` kernel) plus a mel-filter matmul, and folds it
   directly into the traced graph — verified to match the reference NumPy pipeline to ~1e-5 dB. The
   exported `ClapAudioEncoder.mlpackage` therefore takes a **raw waveform**, not a spectrogram.
2. **A Swin-Transformer audio tower (HTSAT) with a lot of `.shape`-derived reshapes.** CLAP reshapes
   the spectrogram into an "image", partitions it into attention windows, merges patches between
   stages, etc. — and PyTorch's tracer represents each of those reshape targets as a *computed*
   `aten::size`/`aten::Int` chain rather than a literal constant. For most models `coremltools` folds
   these back into constants; for this specific op combination (our custom STFT feeding into HTSAT's
   Swin blocks) its constant-folding hits a bug (`only 0-dimensional arrays can be converted to Python
   scalars`) and conversion fails partway through. Since this export always uses a **fixed 10-second
   window at 48kHz**, every one of those shapes is actually known in advance — so the fix is to
   monkey-patch `reshape_mel2img`, `ClapAudioLayer.forward`, `get_attn_mask`,
   `ClapAudioSelfAttention.forward` and `ClapAudioPatchMerging.forward` to use literal Python ints
   instead of tensor `.shape` reads. Verified numerically identical (cosine sim 1.0) to the
   unpatched model before it's ever traced.
3. **FP16 compute precision is unsafe for this model.** Measured, not assumed: exporting with
   `compute_precision=FLOAT16` (the setting used for CLIP) gives a cosine similarity as low as **0.85**
   against the FP32 reference on some tracks for the audio encoder, and **0.02–0.19** (i.e. broken) for
   the text encoder. `compute_precision=FLOAT32` matches the reference to 5 decimal places on every
   sample tried. Both encoders are therefore exported at FLOAT32.
4. **INT8 weight-only quantization crashes CoreML's Metal/ANE graph compiler on this Mac** (`Error:
   MLIR pass manager failed` in `MetalPerformanceShadersGraph`) for *both* encoders — reproduced
   consistently, not a fluke. It only runs under `ComputeUnit.CPU_ONLY`. Given that's a real crash risk
   on-device and this pass is being deferred to the iOS integration step anyway, this notebook ships
   unquantized FLOAT32 `.mlpackage`s and documents the finding rather than silently shipping something
   that might crash — see the **Quantization status** note near the end for what to try next.

**Env setup (uv), run from `trial_models/`:**
```bash
uv sync
uv run --with jupyter jupyter lab export_for_ios/export_clap.ipynb
```

In [1]:
import io
import math
import types
from pathlib import Path

import coremltools as ct
import librosa
import numpy as np
import torch
import torch.nn as nn

OUTPUT_DIR = Path.cwd() / "exported_models"
OUTPUT_DIR.mkdir(exist_ok=True)
AUDIO_DIR = Path.cwd().parent / "audios"

torch.manual_seed(0)
print("torch:", torch.__version__, " coremltools:", ct.__version__)

scikit-learn version 1.9.0 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.


Torch version 2.14.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.7.0 is the most recent version that has been tested.


torch: 2.14.0  coremltools: 9.0


## Load CLAP

`laion/clap-htsat-unfused` — dual-tower HTSAT (audio) + RoBERTa (text) model,
`enable_fusion=False` (confirmed below), which is what makes the Swin patches in this notebook
sufficient — `is_longer`/fusion-specific code paths are simply never exercised.

In [2]:
from transformers import ClapModel, ClapProcessor
from transformers.audio_utils import window_function

CLAP_MODEL_ID = "laion/clap-htsat-unfused"
clap_model = ClapModel.from_pretrained(CLAP_MODEL_ID).eval()
clap_processor = ClapProcessor.from_pretrained(CLAP_MODEL_ID)
for p in clap_model.parameters():
    p.requires_grad_(False)

audio_config = clap_model.config.audio_config
fe = clap_processor.feature_extractor
print(f"enable_fusion={audio_config.enable_fusion} projection_dim={clap_model.config.projection_dim}")
print(f"sample_rate={fe.sampling_rate} n_fft={fe.fft_window_size} hop={fe.hop_length} "
      f"max_length_s={fe.max_length_s} n_mels={fe.feature_size}")

enable_fusion=False projection_dim=512
sample_rate=48000 n_fft=1024 hop=480 max_length_s=10 n_mels=64


## Part 1 — Audio encoder

### STFT-as-Conv1d + mel filterbank

`ClapFeatureExtractor._np_extract_fbank_features` computes, in NumPy: a Hann-windowed STFT
(`n_fft=1024`, `hop=480`, center-padded with reflection), a power spectrogram, a Slaney mel
filterbank matmul, then `10*log10(clip(x, 1e-10))`. Everything here is either a **fixed constant**
(the Hann window, the mel filter matrix — both pulled directly from the already-instantiated
`ClapFeatureExtractor` so they're guaranteed identical to the reference) or a **linear operator**
(the DFT itself), so the whole thing becomes: pad → two `Conv1d`s (cosine/sine basis × window,
one conv per real/imaginary component) → power → mel matmul → log10.

In [3]:
n_fft, hop = fe.fft_window_size, fe.hop_length
hann_window = window_function(n_fft, "hann").astype(np.float64)

sample_n = np.arange(n_fft)
freq_k = np.arange(n_fft // 2 + 1)  # onesided rfft -> 513 bins
angle = 2 * np.pi * np.outer(freq_k, sample_n) / n_fft
cos_basis = (np.cos(angle) * hann_window[None, :]).astype(np.float32)   # (513, 1024)
sin_basis = (-np.sin(angle) * hann_window[None, :]).astype(np.float32)  # (513, 1024)
mel_filters_slaney = fe.mel_filters_slaney.astype(np.float32)           # (513, 64), from the HF feature extractor itself

print("cos/sin basis:", cos_basis.shape, " mel filters:", mel_filters_slaney.shape)

cos/sin basis: (513, 1024)  mel filters: (513, 64)


### `reshape_mel2img`'s bicubic interpolation, as a matmul

The HTSAT audio encoder resizes the spectrogram's time axis from 1001 frames to 1024
(`spec_size * freq_ratio`) via `F.interpolate(..., mode="bicubic", align_corners=True)` — an op
`coremltools` doesn't implement (`NotImplementedError: ... 'upsample_bicubic2d' ...`). For a *fixed*
1001→1024 resize, bicubic-with-align_corners is a linear operator (each output sample is a fixed
weighted sum of input samples), so its exact weight matrix is captured once here by feeding unit
impulses through the real `F.interpolate` call — not approximated, not re-derived by hand — and
reused as a plain matmul at export time (verified to match `F.interpolate` to ~1e-7 max abs diff on
random input below).

In [4]:
SPEC_WIDTH, SPEC_HEIGHT = 1024, 64  # audio_config.spec_size * freq_ratio, spec_size // freq_ratio
TIME_FRAMES = 1001  # frames for a 10s/48kHz window with n_fft=1024, hop=480, center=True

time_interp_matrix = torch.zeros(SPEC_WIDTH, TIME_FRAMES)
for i in range(TIME_FRAMES):
    impulse = torch.zeros(1, 1, TIME_FRAMES, SPEC_HEIGHT)
    impulse[0, 0, i, :] = 1.0
    resized = nn.functional.interpolate(impulse, (SPEC_WIDTH, SPEC_HEIGHT), mode="bicubic", align_corners=True)
    time_interp_matrix[:, i] = resized[0, 0, :, 0]  # freq axis is a no-op here (64 -> 64), see notebook text

# sanity check against the real op on random input
_probe = torch.randn(1, 1, TIME_FRAMES, SPEC_HEIGHT)
_ref = nn.functional.interpolate(_probe, (SPEC_WIDTH, SPEC_HEIGHT), mode="bicubic", align_corners=True)
_mine = torch.einsum("ot,bcte->bcoe", time_interp_matrix, _probe)
print("bicubic-as-matmul max abs diff:", (_ref - _mine).abs().max().item())

bicubic-as-matmul max abs diff: 2.384185791015625e-07


### Literal-shape patches for the Swin (HTSAT) blocks

Everything below replaces a `.shape`/`.size()` read with the equivalent literal Python int — always
safe here because the whole model only ever sees one fixed input shape (10s @ 48kHz → a 1×1×256×256
"image" after `reshape_mel2img`). Correctness is checked immediately after (cosine sim vs. the
unpatched model), not assumed.

In [5]:
from transformers.models.clap.modeling_clap import (
    ClapAudioLayer,
    ClapAudioPatchMerging,
    ClapAudioSelfAttention,
)


def reshape_mel2img_patched(self, normalized_input_features):
    x = torch.einsum("ot,bcte->bcoe", self._time_interp_matrix, normalized_input_features)  # (1,1,1024,64)
    x = x.reshape(1, self.freq_ratio, SPEC_WIDTH // self.freq_ratio, SPEC_HEIGHT)
    x = x.permute(0, 1, 3, 2).contiguous()
    x = x.reshape(1, 1, SPEC_HEIGHT * self.freq_ratio, SPEC_WIDTH // self.freq_ratio)
    return x


def get_attn_mask_patched(self, height, width, dtype, device):
    if self.shift_size > 0:
        ws = self.window_size
        img_mask = torch.zeros((1, height, width, 1), dtype=dtype, device=device)
        h_slices = (slice(0, -ws), slice(-ws, -self.shift_size), slice(-self.shift_size, None))
        w_slices = (slice(0, -ws), slice(-ws, -self.shift_size), slice(-self.shift_size, None))
        count = 0
        for h in h_slices:
            for w in w_slices:
                img_mask[:, h, w, :] = count
                count += 1
        mw = img_mask.view(1, height // ws, ws, width // ws, ws, 1)
        mw = mw.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, ws, ws, 1)
        mask_windows = mw.view(-1, ws * ws)
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
    else:
        attn_mask = None
    return attn_mask


def clap_audio_layer_forward_patched(self, hidden_states, input_dimensions, head_mask=None,
                                      output_attentions=False, always_partition=False):
    if not always_partition:
        self.set_shift_and_window_size(input_dimensions)
    height, width = input_dimensions
    channels = self.layernorm_before.normalized_shape[0]
    shortcut = hidden_states

    hidden_states = self.layernorm_before(hidden_states)
    hidden_states = hidden_states.view(1, height, width, channels)
    hidden_states, pad_values = self.maybe_pad(hidden_states, height, width)
    height_pad = height + pad_values[5]
    width_pad = width + pad_values[3]

    if self.shift_size > 0:
        shifted_hidden_states = torch.roll(hidden_states, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
    else:
        shifted_hidden_states = hidden_states

    ws = self.window_size
    hsw = shifted_hidden_states.view(1, height_pad // ws, ws, width_pad // ws, ws, channels)
    hsw = hsw.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, ws, ws, channels)
    hidden_states_windows = hsw.view(-1, ws * ws, channels)

    attn_mask = self.get_attn_mask(height_pad, width_pad, dtype=hidden_states.dtype, device=hidden_states_windows.device)
    # ClapAudioSelfAttention needs the window count too (see below) -- inject it rather than
    # re-derive it from `.shape` inside attention, which is what actually breaks conversion.
    self.attention.self._num_windows_literal = (height_pad // ws) * (width_pad // ws)
    attention_outputs = self.attention(hidden_states_windows, attn_mask, head_mask, output_attentions=output_attentions)
    attention_output = attention_outputs[0]

    aw = attention_output.view(-1, ws, ws, channels)
    aw = aw.view(-1, height_pad // ws, width_pad // ws, ws, ws, channels)
    aw = aw.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, height_pad, width_pad, channels)
    attention_windows = (
        torch.roll(aw, shifts=(self.shift_size, self.shift_size), dims=(1, 2)) if self.shift_size > 0 else aw
    )

    was_padded = pad_values[3] > 0 or pad_values[5] > 0
    if was_padded:
        attention_windows = attention_windows[:, :height, :width, :].contiguous()
    attention_windows = attention_windows.view(1, height * width, channels)

    hidden_states = shortcut + self.drop_path(attention_windows)
    layer_output = self.layernorm_after(hidden_states)
    layer_output = self.intermediate(layer_output)
    layer_output = hidden_states + self.output(layer_output)
    return (layer_output, attention_outputs[1]) if output_attentions else (layer_output,)


def clap_self_attention_forward_patched(self, hidden_states, attention_mask=None, head_mask=None,
                                         output_attentions=False):
    num_windows = self._num_windows_literal  # set by clap_audio_layer_forward_patched
    dim = self.window_size[0] * self.window_size[1]

    def split_heads(t):
        return t.view(num_windows, dim, self.num_attention_heads, self.attention_head_size).permute(0, 2, 1, 3)

    query_layer = split_heads(self.query(hidden_states))
    key_layer = split_heads(self.key(hidden_states))
    value_layer = split_heads(self.value(hidden_states))

    attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
    attention_scores = attention_scores / math.sqrt(self.attention_head_size)

    relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)]
    relative_position_bias = relative_position_bias.view(dim, dim, -1).permute(2, 0, 1).contiguous()
    attention_scores = attention_scores + relative_position_bias.unsqueeze(0)

    if attention_mask is not None:
        attention_scores = attention_scores.view(1, num_windows, self.num_attention_heads, dim, dim)
        attention_scores = attention_scores + attention_mask.unsqueeze(1).unsqueeze(0)
        attention_scores = attention_scores.view(-1, self.num_attention_heads, dim, dim)

    attention_probs = nn.functional.softmax(attention_scores, dim=-1)
    attention_probs = self.dropout(attention_probs)
    if head_mask is not None:
        attention_probs = attention_probs * head_mask

    context_layer = torch.matmul(attention_probs, value_layer)
    context_layer = context_layer.permute(0, 2, 1, 3).contiguous().view(num_windows, dim, self.all_head_size)
    return (context_layer, attention_probs) if output_attentions else (context_layer,)


def clap_patch_merging_forward_patched(self, input_feature, input_dimensions):
    height, width = input_dimensions
    num_channels = self.dim
    input_feature = input_feature.view(1, height, width, num_channels)
    input_feature = self.maybe_pad(input_feature, height, width)
    f0, f1 = input_feature[:, 0::2, 0::2, :], input_feature[:, 1::2, 0::2, :]
    f2, f3 = input_feature[:, 0::2, 1::2, :], input_feature[:, 1::2, 1::2, :]
    input_feature = torch.cat([f0, f1, f2, f3], -1)
    input_feature = input_feature.view(1, -1, 4 * num_channels)
    input_feature = self.norm(input_feature)
    return self.reduction(input_feature)


def clap_audio_encoder_forward_patched(self, input_features, is_longer=None, head_mask=None,
                                        output_attentions=False, output_hidden_states=False,
                                        output_hidden_states_before_downsampling=False,
                                        always_partition=False, return_dict=True):
    input_features = input_features.transpose(1, 3)
    normalized_input_features = self.batch_norm(input_features)
    normalized_input_features = normalized_input_features.transpose(1, 3)

    hidden_states = self.reshape_mel2img(normalized_input_features)  # (1,1,256,256) by construction
    frames_num = SPEC_HEIGHT * self.freq_ratio  # 256, matches reshape_mel2img's fixed output
    hidden_states = self.patch_embed(hidden_states, None)

    for i, layer_module in enumerate(self.layers):
        input_dimensions = self.input_resolutions[i]
        layer_outputs = layer_module(hidden_states, input_dimensions, None, False, always_partition)
        hidden_states = layer_outputs[0]

    last_hidden_state = self.norm(hidden_states)
    n_channels = self.num_features
    freq_shape = frames_num // (2 ** (len(self.depths) - 1)) // self.patch_stride[0]
    temporal_shape = frames_num // (2 ** (len(self.depths) - 1)) // self.patch_stride[1]
    last_hidden_state = (
        last_hidden_state.permute(0, 2, 1).contiguous().reshape(1, n_channels, freq_shape, temporal_shape)
    )
    c_freq_bin = freq_shape // self.freq_ratio
    last_hidden_state = last_hidden_state.reshape(1, n_channels, freq_shape // c_freq_bin, c_freq_bin, temporal_shape)
    last_hidden_state = (
        last_hidden_state.permute(0, 1, 3, 2, 4).contiguous().reshape(1, n_channels, c_freq_bin, -1)
    )
    latent_output = self.avgpool(torch.flatten(last_hidden_state, 2))
    return torch.flatten(latent_output, 1)


def apply_clap_audio_coreml_patches(clap, time_interp_matrix):
    audio_encoder = clap.audio_model.audio_encoder
    audio_encoder._time_interp_matrix = time_interp_matrix
    audio_encoder.reshape_mel2img = types.MethodType(reshape_mel2img_patched, audio_encoder)
    audio_encoder.forward = types.MethodType(clap_audio_encoder_forward_patched, audio_encoder)
    for m in audio_encoder.modules():
        if isinstance(m, ClapAudioLayer):
            m.forward = types.MethodType(clap_audio_layer_forward_patched, m)
            m.get_attn_mask = types.MethodType(get_attn_mask_patched, m)
        elif isinstance(m, ClapAudioPatchMerging):
            m.forward = types.MethodType(clap_patch_merging_forward_patched, m)
        elif isinstance(m, ClapAudioSelfAttention):
            m.forward = types.MethodType(clap_self_attention_forward_patched, m)

In [6]:
AUDIO_SR = fe.sampling_rate       # 48000
AUDIO_WINDOW_S = fe.max_length_s  # 10
AUDIO_WINDOW_SAMPLES = AUDIO_SR * AUDIO_WINDOW_S  # 480_000

sample_path = sorted(AUDIO_DIR.glob("*.mp3"))[0]
wav, _ = librosa.load(sample_path, sr=AUDIO_SR, mono=True, duration=AUDIO_WINDOW_S)
wav = wav[:AUDIO_WINDOW_SAMPLES]
wav = np.pad(wav, (0, AUDIO_WINDOW_SAMPLES - len(wav))).astype(np.float64)

reference_inputs = clap_processor(audios=wav, sampling_rate=AUDIO_SR, return_tensors="pt")
with torch.no_grad():
    reference_audio_embed = clap_model.get_audio_features(**reference_inputs)

apply_clap_audio_coreml_patches(clap_model, time_interp_matrix)

with torch.no_grad():
    patched_pooled = clap_model.audio_model.audio_encoder(input_features=reference_inputs["input_features"], is_longer=None)
    patched_audio_embed = clap_model.audio_projection(patched_pooled)
    patched_audio_embed = nn.functional.normalize(patched_audio_embed, dim=-1)

cos = nn.functional.cosine_similarity(reference_audio_embed, patched_audio_embed).item()
print(f"patched (eager) vs unpatched HF reference cosine sim: {cos:.7f}")

patched (eager) vs unpatched HF reference cosine sim: 0.9999999


### Trace + convert

The exported model's only input is a raw **`waveform`**: `[1, 480000]` float32 samples at 48kHz
(exactly 10.0s, mono). It runs the Conv1d-STFT front end above, then the (now literal-shape) HTSAT
audio tower, then CLAP's audio projection, then L2-normalizes — a self-contained
waveform → 512-d embedding model, no external preprocessing needed on the iOS side.

In [7]:
class ClapAudioEncoderExport(nn.Module):
    def __init__(self, clap, cos_basis, sin_basis, mel_filters, n_fft, hop):
        super().__init__()
        self.audio_encoder = clap.audio_model.audio_encoder
        self.audio_projection = clap.audio_projection
        self.register_buffer("cos_w", torch.tensor(cos_basis).unsqueeze(1))
        self.register_buffer("sin_w", torch.tensor(sin_basis).unsqueeze(1))
        self.register_buffer("mel_filters_t", torch.tensor(mel_filters))
        self.n_fft, self.hop = n_fft, hop

    def forward(self, waveform):
        x = waveform.unsqueeze(1)
        pad = self.n_fft // 2
        x = nn.functional.pad(x, (pad, pad), mode="reflect")
        real = nn.functional.conv1d(x, self.cos_w, stride=self.hop)
        imag = nn.functional.conv1d(x, self.sin_w, stride=self.hop)
        power = real**2 + imag**2
        mel = torch.clamp(torch.einsum("fm,bft->bmt", self.mel_filters_t, power), min=1e-10)
        log_mel = 10.0 * torch.log10(mel)
        input_features = log_mel.transpose(1, 2).unsqueeze(1)
        pooled = self.audio_encoder(input_features=input_features, is_longer=None)
        embeds = self.audio_projection(pooled)
        return embeds / embeds.norm(p=2, dim=-1, keepdim=True)


audio_encoder_wrapper = ClapAudioEncoderExport(
    clap_model, cos_basis, sin_basis, mel_filters_slaney, n_fft, hop
).eval()

dummy_waveform = torch.zeros(1, AUDIO_WINDOW_SAMPLES)
with torch.no_grad():
    traced_audio_encoder = torch.jit.trace(audio_encoder_wrapper, (dummy_waveform,))
print("trace OK")

/Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/.venv/lib/python3.12/site-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/.venv/lib/python3.12/site-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/.venv/lib/python3.12/site-packages/transformers/models/clap/modeling_clap.py:381: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if height != self.img_size[0] or width != self.img_size[1]:
/Users

/Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/.venv/lib/python3.12/site-packages/transformers/models/clap/modeling_clap.py:592: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if min(input_resolution) <= self.window_size:


trace OK


In [8]:
clap_audio_model = ct.convert(
    traced_audio_encoder,
    inputs=[ct.TensorType(name="waveform", shape=(1, AUDIO_WINDOW_SAMPLES))],
    outputs=[ct.TensorType(name="audio_embeds")],
    convert_to="mlprogram",
    minimum_deployment_target=ct.target.iOS16,
    compute_precision=ct.precision.FLOAT32,  # see "Why CLAP needs..." above -- FLOAT16 is measurably unsafe here
)
clap_audio_model.save(str(OUTPUT_DIR / "ClapAudioEncoder.mlpackage"))
print("saved ClapAudioEncoder.mlpackage")

Converting PyTorch Frontend ==> MIL Ops:   0%|          | 0/1584 [00:00<?, ? ops/s]

Saving value type of int64 into a builtin type of int32, might lose precision!


Saving value type of int64 into a builtin type of int32, might lose precision!


Saving value type of int64 into a builtin type of int32, might lose precision!


Saving value type of int64 into a builtin type of int32, might lose precision!


...

Running MIL default pipeline:  99%|█████████▉| 91/92 [00:01<00:00, 90.00 passes/s]

Running MIL default pipeline: 100%|██████████| 92/92 [00:01<00:00, 78.74 passes/s]

Running MIL backend_mlprogram pipeline:   0%|          | 0/12 [00:00<?, ? passes/s]

Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 202.75 passes/s]

saved ClapAudioEncoder.mlpackage


### Verify: PyTorch vs. the exported CoreML model, on every sample track

Each track's first 10 seconds (the exported model's fixed window) run through both the original
PyTorch `ClapModel.get_audio_features` and the exported CoreML model.

In [9]:
audio_paths = sorted(AUDIO_DIR.glob("*.mp3"))
print(f"Found {len(audio_paths)} sample tracks: {[p.name for p in audio_paths]}")

audio_embeds_by_path = {}
for path in audio_paths:
    w, _ = librosa.load(path, sr=AUDIO_SR, mono=True, duration=AUDIO_WINDOW_S)
    w = w[:AUDIO_WINDOW_SAMPLES]
    w = np.pad(w, (0, max(0, AUDIO_WINDOW_SAMPLES - len(w)))).astype(np.float64)

    # NOTE: `clap_model.get_audio_features` can no longer serve as the reference here -- the
    # patches above mutated `clap_model.audio_model.audio_encoder.forward` in place. Use the
    # (already-verified-equivalent-to-unpatched, see cosine sim above) eager PyTorch wrapper instead,
    # so this loop is checking trace+convert fidelity specifically, not the patches again.
    with torch.no_grad():
        ref_embed = audio_encoder_wrapper(torch.tensor(w, dtype=torch.float32).unsqueeze(0)).numpy()[0]

    coreml_out = clap_audio_model.predict({"waveform": w.astype(np.float32)[None, :]})
    coreml_embed = coreml_out["audio_embeds"][0]
    audio_embeds_by_path[path.name] = coreml_embed

    cos = np.dot(ref_embed, coreml_embed) / (np.linalg.norm(ref_embed) * np.linalg.norm(coreml_embed))
    print(f"{path.name:35s} cosine sim (PyTorch fp32 vs CoreML fp32): {cos:.5f}")

Found 9 sample tracks: ['Cheek To Cheek.mp3', 'Cherry Wine.mp3', 'Clementine.mp3', 'From The Start.mp3', 'Happy!.mp3', 'I Hope to Be Around.mp3', 'Numb.mp3', "Regent's Park.mp3", 'The Most Beautiful Thing.mp3']
Cheek To Cheek.mp3                  cosine sim (PyTorch fp32 vs CoreML fp32): 1.00000
Cherry Wine.mp3                     cosine sim (PyTorch fp32 vs CoreML fp32): 1.00000


Clementine.mp3                      cosine sim (PyTorch fp32 vs CoreML fp32): 1.00000
From The Start.mp3                  cosine sim (PyTorch fp32 vs CoreML fp32): 1.00000
Happy!.mp3                          cosine sim (PyTorch fp32 vs CoreML fp32): 1.00000
I Hope to Be Around.mp3             cosine sim (PyTorch fp32 vs CoreML fp32): 1.00000
Numb.mp3                            cosine sim (PyTorch fp32 vs CoreML fp32): 1.00000


Regent's Park.mp3                   cosine sim (PyTorch fp32 vs CoreML fp32): 1.00000
The Most Beautiful Thing.mp3        cosine sim (PyTorch fp32 vs CoreML fp32): 1.00000


## Part 2 — Text encoder

CLAP's text tower is a plain RoBERTa encoder — standard-enough that it traces and converts cleanly
with no patching needed (unlike the audio side). `get_text_features` already L2-normalizes
internally. FLOAT32 compute precision is used here too — FLOAT16 was measured to produce
**0.02–0.19 cosine similarity** against the reference (i.e. broken, not just imprecise), almost
certainly RoBERTa's `layer_norm_eps=1e-12` underflowing in fp16 arithmetic on the padded positions.

In [10]:
TEXT_SEQ_LEN = 64  # fixed sequence length; CLAP/RoBERTa supports up to 514 but queries are short


class ClapTextEncoderExport(nn.Module):
    def __init__(self, clap):
        super().__init__()
        self.text_model = clap.text_model
        self.text_projection = clap.text_projection

    def forward(self, input_ids, attention_mask):
        out = self.text_model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        feats = self.text_projection(out.pooler_output)
        return feats / feats.norm(p=2, dim=-1, keepdim=True)


text_encoder_wrapper = ClapTextEncoderExport(clap_model).eval()

dummy_ids = torch.zeros(1, TEXT_SEQ_LEN, dtype=torch.long)
dummy_mask = torch.ones(1, TEXT_SEQ_LEN, dtype=torch.long)
with torch.no_grad():
    traced_text_encoder = torch.jit.trace(text_encoder_wrapper, (dummy_ids, dummy_mask))
print("trace OK")

/Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/.venv/lib/python3.12/site-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/.venv/lib/python3.12/site-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


trace OK


In [11]:
clap_text_model = ct.convert(
    traced_text_encoder,
    inputs=[
        ct.TensorType(name="input_ids", shape=(1, TEXT_SEQ_LEN), dtype=np.int32),
        ct.TensorType(name="attention_mask", shape=(1, TEXT_SEQ_LEN), dtype=np.int32),
    ],
    outputs=[ct.TensorType(name="text_embeds")],
    convert_to="mlprogram",
    minimum_deployment_target=ct.target.iOS16,
    compute_precision=ct.precision.FLOAT32,
)
clap_text_model.save(str(OUTPUT_DIR / "ClapTextEncoder.mlpackage"))
print("saved ClapTextEncoder.mlpackage")

Converting PyTorch Frontend ==> MIL Ops:   0%|          | 0/641 [00:00<?, ? ops/s]

Core ML embedding (gather) layer does not support any inputs besides the weights and indices. Those given will be ignored.


Core ML embedding (gather) layer does not support any inputs besides the weights and indices. Those given will be ignored.


Converting PyTorch Frontend ==> MIL Ops: 100%|█████████▉| 640/641 [00:00<00:00, 13881.31 ops/s]

Running MIL frontend_pytorch pipeline:   0%|          | 0/5 [00:00<?, ? passes/s]

...

Running MIL default pipeline:  97%|█████████▋| 89/92 [00:00<00:00, 241.69 passes/s]

Running MIL default pipeline: 100%|██████████| 92/92 [00:00<00:00, 241.26 passes/s]

Running MIL backend_mlprogram pipeline:   0%|          | 0/12 [00:00<?, ? passes/s]

Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 445.61 passes/s]

saved ClapTextEncoder.mlpackage


In [12]:
retrieval_queries = [
    "an upbeat, danceable pop song",
    "a slow, emotional piano ballad",
    "a relaxing acoustic guitar tune",
    "a melancholic, introspective song",
    "a warm, romantic jazz standard",
    "a high-energy rock anthem",
]

query_toks = clap_processor.tokenizer(
    retrieval_queries, padding="max_length", max_length=TEXT_SEQ_LEN, truncation=True, return_tensors="pt"
)
with torch.no_grad():
    ref_text_embeds = clap_model.get_text_features(**query_toks)

# The exported model has a fixed [1, 64] input shape (see iOS integration notes), so predict
# one query at a time -- matching how the app will actually call it (one interactive query),
# rather than tracing/converting a batch-flexible shape it will never use on-device.
text_embeds_by_query = {}
for i, q in enumerate(retrieval_queries):
    coreml_out = clap_text_model.predict({
        "input_ids": query_toks["input_ids"][i:i+1].numpy().astype(np.int32),
        "attention_mask": query_toks["attention_mask"][i:i+1].numpy().astype(np.int32),
    })
    a, b = ref_text_embeds.numpy()[i], coreml_out["text_embeds"][0]
    text_embeds_by_query[q] = b
    cos = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    print(f"{q!r:38s} cosine sim (PyTorch fp32 vs CoreML fp32): {cos:.5f}")

'an upbeat, danceable pop song'        cosine sim (PyTorch fp32 vs CoreML fp32): 1.00000
'a slow, emotional piano ballad'       cosine sim (PyTorch fp32 vs CoreML fp32): 1.00000
'a relaxing acoustic guitar tune'      cosine sim (PyTorch fp32 vs CoreML fp32): 1.00000
'a melancholic, introspective song'    cosine sim (PyTorch fp32 vs CoreML fp32): 1.00000
'a warm, romantic jazz standard'       cosine sim (PyTorch fp32 vs CoreML fp32): 1.00000
'a high-energy rock anthem'            cosine sim (PyTorch fp32 vs CoreML fp32): 1.00000


### Sanity check: cross-modal retrieval using only the exported CoreML models

Embed every sample track (first 10s) and every query with the two `.mlpackage`s above, rank tracks
per query by cosine similarity — the exact retrieval flow the iOS app will run.

In [13]:
for query, q_embed in text_embeds_by_query.items():
    ranked = sorted(audio_embeds_by_path.items(), key=lambda kv: -float(np.dot(q_embed, kv[1])))
    print(f"query: {query!r}")
    for name, embed in ranked:
        print(f"    {float(np.dot(q_embed, embed)):.4f}  {name}")

query: 'an upbeat, danceable pop song'
    0.6384  I Hope to Be Around.mp3
    0.5864  Clementine.mp3
    0.5770  Numb.mp3
    0.5581  The Most Beautiful Thing.mp3
    0.5032  Cheek To Cheek.mp3
    0.4808  Happy!.mp3
    0.4751  Cherry Wine.mp3
    0.4593  From The Start.mp3
    0.3777  Regent's Park.mp3
query: 'a slow, emotional piano ballad'
    0.3774  I Hope to Be Around.mp3
    0.3771  Regent's Park.mp3
    0.3719  Clementine.mp3
    0.3399  Numb.mp3
    0.3381  The Most Beautiful Thing.mp3
    0.3225  Happy!.mp3
    0.2607  Cheek To Cheek.mp3
    0.2126  From The Start.mp3
    0.1932  Cherry Wine.mp3
query: 'a relaxing acoustic guitar tune'
    0.4883  Clementine.mp3
    0.4761  The Most Beautiful Thing.mp3
    0.4565  From The Start.mp3
    0.4532  Regent's Park.mp3
    0.4232  I Hope to Be Around.mp3
    0.3986  Happy!.mp3
    0.3495  Cheek To Cheek.mp3
    0.2766  Cherry Wine.mp3
    0.2557  Numb.mp3
query: 'a melancholic, introspective song'
    0.4546  Clementine.mp3
    0.

## Tokenizer assets for on-device use

The text encoder operates on token ids, not raw text — iOS needs CLAP's RoBERTa-BPE vocab to
tokenize queries (e.g. via [swift-transformers](https://github.com/huggingface/swift-transformers)'
`Tokenizers` package).

In [14]:
clap_tokenizer_dir = OUTPUT_DIR / "clap_tokenizer"
clap_processor.tokenizer.save_pretrained(clap_tokenizer_dir)
print("CLAP tokenizer files:", sorted(p.name for p in clap_tokenizer_dir.iterdir()))

CLAP tokenizer files: ['merges.txt', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


## Quantization status

**Not shipped in this export** — both attempts below were measured to be unsafe, not just skipped
for convenience:

- **`compute_precision=FLOAT16`**: audio encoder cosine similarity dropped as low as 0.85 on some
  tracks; text encoder dropped to 0.02–0.19 (broken). Both encoders ship at FLOAT32.
- **INT8 weight-only quantization** (`coremltools.optimize.coreml.linear_quantize_weights`, default
  compute units): numerically fine on CPU, but crashes CoreML's Metal/ANE graph compiler on this Mac
  with `Error: MLIR pass manager failed` in `MetalPerformanceShadersGraph` — reproduced for *both*
  encoders, so it's not specific to the custom audio-encoder ops. Runs fine forced to
  `ComputeUnit.CPU_ONLY`, which defeats the point of quantizing for speed.

**Next steps to try during iOS integration** (once real-device testing is possible, since this may be
a macOS/Simulator-specific Metal compiler issue rather than a true device-side one):
- Re-test INT8 weight quantization directly on a physical iPhone/iPad.
- Try palettization (`cto.OpPalettizerConfig`) instead of linear quantization.
- Mixed precision: keep LayerNorm/softmax in FP32 and cast only the large `Linear` matmuls to FP16,
  rather than an all-or-nothing `compute_precision` switch.

## Summary

In [15]:
def dir_size_mb(path):
    if path.is_file():
        return path.stat().st_size / 1e6
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e6


print(f"{'artifact':28s} {'size (MB)':>10s}")
for item in sorted(OUTPUT_DIR.iterdir()):
    if item.name.startswith("Clap") or item.name == "clap_tokenizer":
        print(f"{item.name:28s} {dir_size_mb(item):10.1f}")

artifact                      size (MB)
ClapAudioEncoder.mlpackage        124.4
ClapTextEncoder.mlpackage         501.5
clap_tokenizer                      4.8


### iOS integration notes

- **Audio encoder** (`ClapAudioEncoder.mlpackage`): input `waveform`, `[1, 480000]` float32, mono
  audio resampled to **48kHz**, **exactly 10.0 seconds**. For tracks under 10s, right-pad with
  zeros. For tracks over 10s (the normal case for full songs), chunk into consecutive (or a few
  evenly-spaced) 10s windows, embed each, average the embeddings, then L2-renormalize — embed each
  track once at library-scan time and cache the result. Output `audio_embeds` is a 512-d
  already-L2-normalized embedding.
- **Text encoder** (`ClapTextEncoder.mlpackage`): inputs `input_ids`/`attention_mask`, both
  `[1, 64]` int32 — tokenize with `clap_tokenizer/` (RoBERTa BPE), pad to length 64 with the
  tokenizer's own pad token, truncate longer queries. Output `text_embeds` is a 512-d
  already-L2-normalized embedding in the *same* space as `audio_embeds` — rank tracks per query by
  plain dot product.
- **Quantization**: both models currently ship at FLOAT32 (see "Quantization status" above) —
  expect ~120MB (audio) + ~480MB (text, dominated by the 50265×768 RoBERTa embedding table) until
  that's revisited on-device.